# Solar Panel Segmentation - YOLO11m-seg Training & Analysis
This notebook trains YOLO11m-seg on satellite imagery (2021-2025) and evaluates test performance.

**Hardware Recommendation:** Google Colab GPU (T4 / V100 / A100).

## 1. Setup Environment & Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install dependencies
!pip install -q ultralytics roboflow matplotlib opencv-python-headless pyyaml

## 2. Mount Google Drive or Upload Dataset
If your dataset is already prepared (`solar-panels-YOLO.zip` or `solar-panel-exports/YOLO`), you can extract it here.

In [ ]:
import os
from pathlib import Path

# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/solar-panels-YOLO.zip ./solar-panels-YOLO.zip

# Extract YOLO dataset if zipped
if os.path.exists('solar-panels-YOLO.zip'):
    !unzip -q solar-panels-YOLO.zip -d solar-panel-exports/YOLO
    print("Unzipped dataset into solar-panel-exports/YOLO")
elif os.path.exists('solar-panel-exports/YOLO'):
    print("Dataset directory solar-panel-exports/YOLO found.")
else:
    print("Please upload 'solar-panels-YOLO.zip' or clone/copy your 'solar-panel-exports' folder.")

## 3. Verify `data.yaml` Configuration
Make sure `data.yaml` points to the correct paths inside Colab.

In [ ]:
yaml_path = Path("solar-panel-exports/YOLO/data.yaml")
if yaml_path.exists():
    print("--- data.yaml content ---")
    print(yaml_path.read_text())
else:
    print("data.yaml not found at", yaml_path)

## 4. Train YOLO11m-seg

In [ ]:
from ultralytics import YOLO

# Initialize pretrained YOLO11 medium segmentation model
model = YOLO("yolo11m-seg.pt")

# Train with satellite-friendly augmentations
results = model.train(
    data="solar-panel-exports/YOLO/data.yaml",
    epochs=150,
    imgsz=432,
    batch=16,             # Colab T4/A100 has 15-40GB VRAM, batch=16 is great
    patience=30,
    workers=2,
    device=0,
    # Augmentations
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=180,
    flipud=0.5,
    fliplr=0.5,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    # Outputs
    project="runs/solar-seg",
    name="yolo11m-seg",
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

print("Training completed!")

## 5. Evaluate Best Weights on Test Split

In [ ]:
best_weights = "runs/solar-seg/yolo11m-seg/weights/best.pt"
val_model = YOLO(best_weights)

metrics = val_model.val(
    data="solar-panel-exports/YOLO/data.yaml",
    split="test",
    imgsz=432,
    batch=16,
    plots=True,
    project="runs/solar-seg",
    name="yolo11m-seg-test",
)

print("=" * 60)
print(f"Test Set mAP50:    {metrics.seg.map50:.4f}")
print(f"Test Set mAP50-95: {metrics.seg.map:.4f}")
print("=" * 60)

## 6. Download / Save Artifacts to Google Drive

In [ ]:
# Zip runs directory
!zip -r yolo11m-seg-results.zip runs/solar-seg

# Optional: download directly in Colab browser
# from google.colab import files
# files.download('yolo11m-seg-results.zip')